**MERGING THE BASE MODEL WITH TRAINED ADAPTERS**

To be clear in case of the future

Merging into Meta's original 16-bit base (`meta-llama/Meta-Llama-3-8B-Instruct`) rather than the 4-bit checkpoint (`unsloth/...-bnb-4bit`) preserves maximum output quality and mathematical accuracy. The 4-bit base underwent irreversible, lossy quantization; dequantizing it back to 16-bit merely casts truncated values into a larger container without restoring the lost detail. Because LoRA trains high-precision 16-bit delta updates, adding them directly to Meta's original unquantized weights prevents compounding rounding errors and eliminates quantization artifacts from the final model weights.

This approach also streamlines the development and deployment pipelines. Merging against the Meta base uses standard `peft` and `transformers` modules, avoiding the binary, CUDA, and PyTorch runtime conflicts that occur when running Unsloth alongside vLLM. The resulting standalone model is universally compatible across all production inference engines—such as vanilla vLLM, Ollama, and TGI—without requiring `bitsandbytes` or custom loader plugins. It also provides a clean, uncorrupted foundation if you decide to apply optimized inference quantization formats like AWQ, GPTQ, or GGUF later.

In [28]:
# install unsloth and required dependencies for this project

!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unsloth.git"
!pip install --upgrade --no-cache-dir unsloth_zoo bitsandbytes "torchao>=0.16.0"

  Cloning https://github.com/unsloth.git to /tmp/pip-install-8ko2ykik/unsloth_67879e4335c44d0ca8408d0616ee5ebf
  Running command git clone --filter=blob:none --quiet https://github.com/unsloth.git /tmp/pip-install-8ko2ykik/unsloth_67879e4335c44d0ca8408d0616ee5ebf
  remote: Not Found
  fatal: repository 'https://github.com/unsloth.git/' not found
  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com/unsloth.git /tmp/pip-install-8ko2ykik/unsloth_67879e4335c44d0ca8408d0616ee5ebf did not run successfully.
  │ exit code: 128
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× git clone --filter=blob:none --quiet https://github.com/unsloth.git /tmp/pip-install-8ko2ykik/unsloth_67879e4335c44d0ca8408d0616ee5ebf did not run successfully.
│ exit code: 128
╰─> See above for output.

note: This error originates from a subprocess, and is likely not 

In [29]:
! nvidia-smi

Mon Sep 28 00:07:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             28W /   70W |    6933MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [30]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

gpu_count = torch.cuda.device_count()
print(f"Number of GPUs: {gpu_count}")

for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

PyTorch Version: 2.10.0+cu128
CUDA Available: True
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [31]:
import os

base_dir = "/kaggle/input"

print("Does /kaggle/input exist?", os.path.exists(base_dir))
print("Contents of /kaggle/input:", os.listdir(base_dir))

Does /kaggle/input exist? True
Contents of /kaggle/input: ['datasets']


In [32]:
os.listdir(base_dir)

['datasets']

In [33]:
import os

print(os.listdir("/kaggle/input/datasets/paulajibola/lora-finetuned-adapter/lora_adapter_v6"))

['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']


In [34]:
import os

# Replace with the exact directory path identified from your previous cell
adapter_path = "/kaggle/input/datasets/paulajibola/lora-finetuned-adapter/lora_adapter_v6"

# Check if the directory exists
folder_exists = os.path.isdir(adapter_path)
print(f"Folder exists: {folder_exists}")

# Check for adapter_model weights (.safetensors or .bin)
safetensors_path = os.path.join(adapter_path, "adapter_model.safetensors")
bin_path = os.path.join(adapter_path, "adapter_model.bin")

adapter_model_exists = os.path.exists(safetensors_path) or os.path.exists(bin_path)
print(f"adapter_model exists: {adapter_model_exists}")

Folder exists: True
adapter_model exists: True


In [35]:
import torch
import sys


print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [36]:
# load the base model
!pip install -U transformers peft accelerate safetensors

In [37]:
!pip install torchvision

In [38]:
# import torch
import transformers
import peft


# print(f"PyTorch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"PEFT Version: {peft.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print("GPU:", torch.cuda.get_device_name(0))

Transformers Version: 5.17.0
PEFT Version: 0.21.0
CUDA Available: True
GPU: Tesla T4


In [39]:
from pathlib import Path


adapter_path = Path(
    "/kaggle/input/datasets/paulajibola/lora-finetuned-adapter/lora_adapter_v6"
)

print("Adapter folder exists:", adapter_path.exists())

print("\nAdapter files:")
for file in adapter_path.iterdir():
    print(" -", file.name)

Adapter folder exists: True

Adapter files:
 - adapter_model.safetensors
 - adapter_config.json
 - README.md
 - tokenizer.json
 - tokenizer_config.json
 - chat_template.jinja


In [40]:
import json

config_path = "/kaggle/input/datasets/paulajibola/lora-finetuned-adapter/lora_adapter_v6/adapter_config.json"

with open(config_path, "r") as f:
    adapter_config = json.load(f)

print(json.dumps(adapter_config, indent=2))

{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": {
    "base_model_class": "LlamaForCausalLM",
    "parent_library": "transformers.models.llama.modeling_llama",
    "unsloth_fixed": true
  },
  "base_model_name_or_path": "unsloth/llama-3-8b-Instruct-bnb-4bit",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_bias": false,
  "lora_dropout": 0,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "monteclora_config": null,
  "peft_type": "LORA",
  "peft_version": "0.20.0",
  "qalora_group_size": 16,
  "r": 16,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "k_proj"

In [43]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")

print("HF token loaded successfully:", hf_token[:10] + "...")

HF token loaded successfully: hf_MDxvkIp...


In [44]:
# The model used for the finetuning is a 4-bit quantized model, and it can't be merged iwth the adapter
# use an equivalent, non-quantized Llama 3 8B Instruct base model

from huggingface_hub import login

login(token=hf_token)

In [45]:
from huggingface_hub import whoami

try:
    user_info = whoami()
    print("Hugging Face login is working.")
    print("username:", user_info["name"])
except Exception as e:
    print("Hugging Face login is NOT working.")
    print("Error:", e)

Hugging Face login is working.
username: Paul-Ajibola


In [46]:
from transformers import AutoTokenizer

base_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    token=hf_token
)

print("Base model tokenizer loaded successfully!")

Base model tokenizer loaded successfully!


In [47]:
# Check for space left before loading models

import torch
import shutil
import psutil


print("GPU:")
print(torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


print("\nSystem RAM:")
print(f"Total RAM: {psutil.virtual_memory().total / 1024**3:.2f} GB")
print(f"Available RAM: {psutil.virtual_memory().available / 1024**3:.2f} GB")


print("\nDisk:")
disk = shutil.disk_usage("/kaggle/working")
print(f"Free disk: {disk.free / 1024**3:.2f} GB")

GPU:
Tesla T4
GPU memory: 14.56 GB

System RAM:
Total RAM: 31.35 GB
Available RAM: 28.71 GB

Disk:
Free disk: 19.50 GB


In [48]:
import os
import shutil

adapter_path = "/kaggle/input/datasets/paulajibola/lora-finetuned-adapter/lora_adapter_v6"
hf_cache = os.path.expanduser("~/.cache/huggingface")


def get_size_gb(path):
    total = 0
    if os.path.isfile(path):
        total = os.path.getsize(path)
    elif os.path.isdir(path):
        for root, dirs, files in os.walk(path):
            for file in files:
                total += os.path.getsize(os.path.join(root, file))
    return total / (1024**3)

print("Adapter size:")
print(f"{get_size_gb(adapter_path):.2f} GB")

print("\nHugging Face cache:")
if os.path.exists(hf_cache):
    print(f"{get_size_gb(hf_cache):.2f} GB")
else:
    print("No Hugging Face cache found.")

print("\nFree disk space:")
disk = shutil.disk_usage("/kaggle/working")
print(f"{disk.free / (1024**3):.2f} GB")

Adapter size:
0.07 GB

Hugging Face cache:
29.93 GB

Free disk space:
19.50 GB


In [49]:
# LOAD THE BASE MODEL

import torch
from transformers import AutoModelForCausalLM


base_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"


base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    token=hf_token
)

# auto: that should decide how to distribute the model 
# rather than forcing all 8B parameters of the model into the T4

print("Base Model loaded successfully!")
print(base_model.hf_device_map)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Base Model loaded successfully!
{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 'cpu', 'model.layers.30': 'cpu', 'model.layers.31': 'cpu', 'model.norm': 'cpu', 'model.rotary_emb': 'cpu', 'lm_head': 'cpu'}


In [50]:
!pip install -U "torchaudio>=0.16.0"

In [51]:
!pip install --no-cache-dir --force-reinstall "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 39.5 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.18.0
    Uninstalling torchao-0.18.0:
      Successfully uninstalled torchao-0.18.0


In [59]:

# print("TorchAO:", torchao.__version__)
# print("Location:", torchao.__file__)

In [54]:
# LOAD THE ADAPTER

from peft import PeftModel

adapter_path = "/kaggle/input/datasets/paulajibola/lora-finetuned-adapter/lora_adapter_v6"


model_with_adapter = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

print("LoRA adapter loaded successfully!")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


LoRA adapter loaded successfully!


In [55]:
merged_model = model_with_adapter.merge_and_unload(
    safe_merge=True
)


print("LoRA adapter merged successfully!")

LoRA adapter merged successfully!


In [61]:
import shutil

disk = shutil.disk_usage("/kaggle/working")

print(f"Total: {disk.total / (1024**3):.2f} GB")
print(f"Used:  {disk.used / (1024**3):.2f} GB")
print(f"Free:  {disk.free / (1024**3):.2f} GB")

Total: 19.52 GB
Used:  14.97 GB
Free:  4.53 GB


In [63]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        path = os.path.join(root, file)
        size_gb = os.path.getsize(path) / (1024**3)

        if size_gb > 0.5:
            print(f"{size_gb:.2f} GB  →  {path}")

14.96 GB  →  /kaggle/working/support-triage-llama3-8b/model.safetensors


In [65]:
import os

output_dir = "/kaggle/working/support-triage-llama3-8b"

# check out the size of the model
for item in os.listdir(output_dir):
    path = os.path.join(output_dir, item)
    size_mb = os.path.getsize(path) / (1024**2)

    print(f"{item:40} {size_mb:10.2f} MB")

tokenizer.json                                16.41 MB
tokenizer_config.json                          0.00 MB
config.json                                    0.00 MB
chat_template.jinja                            0.00 MB
generation_config.json                         0.00 MB
model.safetensors                          15316.54 MB


In [68]:
import os

model_path = "/kaggle/working/support-triage-llama3-8b/model.safetensors"

if os.path.exists(model_path):
    size_gb = os.path.getsize(model_path) / (1024**3)
    print(f"Model exists: {model_path}")
    print(f"Model size: {size_gb:.2f} GB")
else:
    print("Model file does not exist...")

Model exists: /kaggle/working/support-triage-llama3-8b/model.safetensors
Model size: 14.96 GB


In [66]:
import os
import shutil


output_dir = "/kaggle/working/support-triage-llama3-8b"

os.makedirs(output_dir, exist_ok=True)


merged_model.save_pretrained(
    output_dir,
    safe_serialization=True
)

tokenizer.save_pretrained(output_dir)

print("Merged model and tokenizer saved successfully!")

disk = shutil.disk_usage("/kaggle/working")
print(f"Free disk remaining: {disk.free / (1024**3):.2f} GB")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SafetensorError: Error while serializing: I/O error: No space left on device (os error 28)

In [57]:
# verify the saved model

import os

output_dir = "/kaggle/working/support-triage-llama3-8b"

print("Files in merged model folder:\n")


for file in os.listdir(output_dir):
    file_path = os.path.join(output_dir, file)
    size_mb = os.path.getsize(file_path) / (1024**2)

    print(f"{file:45} {size_mb:.2f} MB")

Files in merged model folder:

tokenizer.json                                16.41 MB
tokenizer_config.json                         0.00 MB
config.json                                   0.00 MB
chat_template.jinja                           0.00 MB
generation_config.json                        0.00 MB
model.safetensors                             15316.54 MB


In [69]:
# test the saved, merged model is loadable independently

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


merged_model_path = "/kaggle/working/support-triage-llama3-8b"

tokenizer = AutoTokenizer.from_pretrained(merged_model_path)

test_model = AutoModelForCausalLM.from_pretrained(
    merged_model_path,
    torch_dtype="auto",
    device_map="auto",
    low_cpu_mem_usage=True
)

print("Merged model loaded successfully!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Merged model loaded successfully!


In [70]:
model_path = "/kaggle/working/support-triage-llama3-8b"

print(os.listdir(model_path))

['tokenizer.json', 'tokenizer_config.json', 'config.json', 'chat_template.jinja', 'generation_config.json', 'model.safetensors']


In [71]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")

print("HF token loaded successfully:", hf_token[:10] + "...")

HF token loaded successfully: hf_MDxvkIp...


In [72]:
# upload the model to the huggingface repo
from huggingface_hub import login

login(token=hf_token)

print("Logged in to Hugging Face successfully!")

Logged in to Hugging Face successfully!


In [73]:
# upload the merged model to huggingface repo

from huggingface_hub import HfApi

api = HfApi(token=hf_token)

repo_id = "Paul-Ajibola/support-triage-llama3"

api.upload_folder(
    folder_path="/kaggle/working/support-triage-llama3-8b",
    repo_id=repo_id,
    repo_type="model"
)

print("Model uploaded successfully!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Model uploaded successfully!


In [75]:
os.path.exists("/kaggle/working/support-triage-llama3-8b")

True